# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nadim7009/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Which pages under-capture clicks, and which of them should a person look at first?** The decision: an SEO editor has time for 20 to 50 page reviews a week and needs an order to read them in. The lane is CTR / engagement opportunity scoring.

In [1]:
import json, pathlib, os
while not os.path.isdir("data/raw") and os.getcwd() != "/": os.chdir("..")
OUT = pathlib.Path("work/outputs"); FIG = pathlib.Path("work/figures")
base = json.load(open(OUT / "baseline_metrics.json"))
model = json.load(open(OUT / "model_metrics.json"))
valid = json.load(open(OUT / "validation_metrics.json"))
play = json.load(open(OUT / "playbook_metrics.json"))
print("Receipts loaded: baseline, model, validation, playbook")

Receipts loaded: baseline, model, validation, playbook


## 2. Data

The starter release of the FlyRank ML Internship dataset, one anonymised table with one row per page and 90-day search and analytics numbers. The pool is pages with a real average position (above 0, up to 20) and at least 500 impressions: 12,023 pages from 28 clients. Pages with no position data and pages below the impression floor were excluded, because their CTR swings on one or two clicks. No client names, domains, URLs or queries are used.

In [2]:
print(f"Pool: {base['pool_pages']:,} pages, {base['clients']} clients")
print(f"Largest client share of the pool: {100 * play['largest_client_share']:.0f}%")

Pool: 12,023 pages, 28 clients
Largest client share of the pool: 37%


## 3. Methodology

- **Target:** observed `ctr` (percent). **Baseline:** median CTR of the page's position tier.
- **Model:** gradient boosting (absolute-error loss, weighted by impressions) on position, impressions, content type, intent, word count and age.
- **Validation:** 5-fold split grouped by client, seed 42. Metric is impressions-weighted mean absolute CTR error.
- **Leakage checks:** `clicks_90d` removed (ctr = clicks / impressions), `client_id` removed, trend fields never used. Adding `clicks_90d` drops the error to 0.045, which is how I know it is a leak.
- **Ranking score:** click gap = max(expected CTR - CTR, 0) / 100 x impressions, from out-of-fold predictions.

## 4. Results (vs baseline)

In [3]:
import pandas as pd
tbl = pd.DataFrame({
    "split": ["random 5-fold", "grouped by client"],
    "tier-median rule": [valid["before_random_split"]["rule_wmae"], valid["after_grouped_split"]["rule_wmae"]],
    "boosting model": [valid["before_random_split"]["model_wmae"], valid["after_grouped_split"]["model_wmae"]],
    "model lower by %": [valid["model_gain_pct_before"], valid["model_gain_pct_after"]]})
print(tbl.to_string(index=False))
print(f"\nGrouped split: model lower in {model['model_better_folds']} of 5 folds; top-50 overlap between rule and model queues: {model['top50_overlap_rule_vs_model']} of 50")

            split  tier-median rule  boosting model  model lower by %
    random 5-fold            0.2232          0.1946              12.8
grouped by client            0.2554          0.2450               4.1

Grouped split: model lower in 4 of 5 folds; top-50 overlap between rule and model queues: 21 of 50


The honest number is the grouped one. A random split makes the model look about three times better than it does on held-out clients.

## 5. Limitations

Observational, one 90-day window, 12,023 pages, 28 clients with one client holding about 37% of the pool. No future window, so nothing here shows that a flagged page stays low or that an edit raises clicks. No query data, so a low CTR may come from branded searches, a featured snippet or an AI answer. The model beats the lookup table by about 4% on held-out clients, which is a small, directional edge. The `top_3` tier is unreliable on this slice. Nothing here says anything about how Google ranks pages.

## 6. Ranked recommendations

In [4]:
print("Archetype counts:", play["archetype_counts"])
print("Action counts:   ", play["action_counts"])
print(f"Editor queue: {play['actionable_pages']:,} pages; high-volume pages routed to an owner: {play['high_volume_pages']}")
print(f"Top 50 review_title_meta pages hold {play['top50_review_gap_share_pct']}% of the pool click gap (a ceiling, not a forecast)")
if (OUT / "action_queue.csv").exists():
    print(pd.read_csv(OUT / "action_queue.csv").head(10).to_string(index=False))
else:
    print("action_queue.csv is gitignored by design; run w07_action_playbook.ipynb to rebuild it.")

Archetype counts: {'over_performing': 5011, 'holding': 2919, 'under_clicking': 2464, 'visible_ignored': 1215, 'high_volume': 414}
Action counts:    {'protect': 5011, 'monitor': 2919, 'review_title_meta': 2464, 'check_indexing_and_intent': 1215, 'human_review_only': 414}
Editor queue: 3,679 pages; high-volume pages routed to an owner: 414
Top 50 review_title_meta pages hold 5.4% of the pool click gap (a ceiling, not a forecast)
 action_rank           content_id position_tier  avg_position  impressions_90d  clicks_90d  ctr  expected_ctr  click_gap       archetype         reason_code                    action
           1 content_c8e9d6ab9013        page_1           9.7           208678           0 0.00         0.202      421.5 visible_ignored zero_clicks_visible check_indexing_and_intent
           2 content_9dbb19a88545        page_1           3.3            38796         117 0.30         0.895      230.8  under_clicking  ctr_below_expected         review_title_meta
           3 content

## 7. Artifacts the paper embeds

In [5]:
for f in ["before_after_split.png", "fold_errors.png", "archetype_counts.png", "staleness_vs_decline.png"]:
    assert (FIG / f).exists(), f"missing figure {f}, rerun w05 to w07"
    print("ok", f)

ok before_after_split.png
ok fold_errors.png
ok archetype_counts.png
ok staleness_vs_decline.png


## Self-check

- [x] Every section above is filled, with the thinking in markdown and the code that backs it
- [x] The notebook runs top to bottom with no errors (it reads the receipts the earlier notebooks write)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit the repo URL on the card.
- [x] The deployed paper has all 9 sections, including the Abstract and Acknowledgments & data credit (link to https://flyrank.ai)